In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]


spark = SparkSession.builder.appName("HealthcarePipeline").getOrCreate()
df = spark.createDataFrame(data, columns)
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
df = df.withColumn("total_bill",df.consultation_fee + (df.tests_count * 1000))
df = df.withColumn("patient_type",when(df.total_bill > 6000, "High").otherwise("Normal"))
df.show()

+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+-----------+----------------+-----------+----------+------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|      6000|      Normal|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|      5000|      Normal|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|      2500|      Normal|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|      7000|        High|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|      8000|        High|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|      6000|      Normal|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|      6000|      Normal|
|     108|

In [0]:
high_value = df.filter(df.total_bill > 6000)
high_value.show()

+--------+------------+---------+----------+----------------+-----------+----------+------------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_bill|patient_type|
+--------+------------+---------+----------+----------------+-----------+----------+------------+
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|      7000|        High|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|      8000|        High|
+--------+------------+---------+----------+----------------+-----------+----------+------------+



In [0]:
dept_summary = df.groupBy("department").agg(sum("total_bill").alias("total_revenue"),avg("total_bill").alias("avg_bill"),count("*").alias("patient_count"))

dept_summary.show()

+-----------+-------------+-----------------+-------------+
| department|total_revenue|         avg_bill|patient_count|
+-----------+-------------+-----------------+-------------+
| Cardiology|        19000|6333.333333333333|            3|
|Orthopedics|        11000|           5500.0|            2|
|Dermatology|         6000|           3000.0|            2|
|  Neurology|         8000|           8000.0|            1|
+-----------+-------------+-----------------+-------------+



In [0]:
dept_summary.orderBy(col("total_revenue").desc()).show()

+-----------+-------------+-----------------+-------------+
| department|total_revenue|         avg_bill|patient_count|
+-----------+-------------+-----------------+-------------+
| Cardiology|        19000|6333.333333333333|            3|
|Orthopedics|        11000|           5500.0|            2|
|  Neurology|         8000|           8000.0|            1|
|Dermatology|         6000|           3000.0|            2|
+-----------+-------------+-----------------+-------------+



In [0]:
df.createOrReplaceTempView("hospital_visits")


In [0]:
%sql
SELECT *FROM hospital_visits WHERE department = 'Cardiology';

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,6000,Normal
104,Priya Nair,Bangalore,Cardiology,5000,2,7000,High
107,Karan Patel,Ahmedabad,Cardiology,5000,1,6000,Normal


In [0]:
%sql
SELECT city,SUM(consultation_fee + (tests_count * 1000)) AS revenue
FROM hospital_visits
GROUP BY city
ORDER BY revenue DESC;

city,revenue
Bangalore,10500
Chennai,8000
Hyderabad,6000
Kolkata,6000
Ahmedabad,6000
Delhi,5000
Mumbai,2500


In [0]:
%sql
SELECT patient_name,consultation_fee + (tests_count * 1000) AS total_bill
FROM hospital_visits
ORDER BY total_bill DESC;

patient_name,total_bill
Vikram Singh,8000
Priya Nair,7000
Arjun Reddy,6000
Ananya Das,6000
Karan Patel,6000
Sneha Kapoor,5000
Meera Iyer,3500
Rahul Sharma,2500


In [0]:
%sql
SELECT department,COUNT(*) AS patient_count
FROM hospital_visits
GROUP BY department;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("hospital_delta")

In [0]:
%sql
INSERT INTO hospital_delta
VALUES (109,'Rohit','Pune','Neurology',6500,2,8500,'High')

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE hospital_delta
SET consultation_fee = 8000
WHERE visit_id = 105

num_affected_rows
1


In [0]:
%sql
DELETE FROM hospital_delta
WHERE visit_id = 103

num_affected_rows
1


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW updates AS
SELECT * FROM VALUES
(105,'Vikram','Chennai','Neurology',9000,2,11000,'High'),
(110,'Neha','Delhi','Cardiology',6000,1,7000,'High')
AS updates(visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type);

MERGE INTO hospital_delta t
USING updates s
ON t.visit_id = s.visit_id

WHEN MATCHED THEN
UPDATE SET t.patient_name = s.patient_name,t.city = s.city,t.department = s.department,t.consultation_fee = s.consultation_fee,t.tests_count = s.tests_count,t.total_bill = s.total_bill,t.patient_type = s.patient_type

WHEN NOT MATCHED THEN
INSERT (visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type)
VALUES (s.visit_id,s.patient_name,s.city,s.department,s.consultation_fee,s.tests_count,s.total_bill,s.patient_type)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
DESCRIBE HISTORY hospital_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-05-04T05:12:51.000Z,143640016069755,azuser5816_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3882680734574324),b2218eaf-c76b-49a0-a763-d3a9fc391664,0504-035505-pghtfzwu-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7249, p25FileSize -> 2820, numDeletionVectorsRemoved -> 1, minFileSize -> 2820, numAddedFiles -> 1, maxFileSize -> 2820, p75FileSize -> 2820, p50FileSize -> 2820, numAddedBytes -> 2820)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T05:12:49.000Z,143640016069755,azuser5816_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#12870L = cast(visit_id#12854 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3882680734574324),b2218eaf-c76b-49a0-a763-d3a9fc391664,0504-035505-pghtfzwu-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4440, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3490, materializeSourceTimeMs -> 294, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1229, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1913)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-04T05:08:55.000Z,143640016069755,azuser5816_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3882680734574324),f30898c8-ef1b-4b59-9e52-3a582757416b,0504-035505-pghtfzwu-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2845, p25FileSize -> 2809, numDeletionVectorsRemoved -> 1, minFileSize -> 2809, numAddedFiles -> 1, maxFileSize -> 2809, p75FileSize -> 2809, p50FileSize -> 2809, numAddedBytes -> 2809)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T05:08:53.000Z,143640016069755,azuser5816_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(visit_id#12445L = 103)""])",null,List(3882680734574324),f30898c8-ef1b-4b59-9e52-3a582757416b,0504-035505-pghtfzwu-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1447, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1095, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 351)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-04T05:07:31.000Z,143640016069755,azuser5816_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3882680734574324),8caa46f6-63dd-41db-b0be-39d0f94752fe,0504-035505-pghtfzwu-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7312, p25FileSize -> 2845, numDeletionVectorsRemoved -> 1, minFileSize -> 2845, numAddedFiles -> 1, maxFileSize -> 2845, p75FileSize -> 2845, p50FileSize -> 2845, numAddedBytes -> 2845)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-04T05:07:29.000Z,143640016069755,azuser5816_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(visit_id#11838L = 105)""])",null,List(3882680734574324),8caa46f6-63dd-41db-b0be-

In [0]:
%sql
SELECT *
FROM hospital_delta VERSION AS OF 1;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,6000,Normal
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,5000,Normal
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2500,Normal
104,Priya Nair,Bangalore,Cardiology,5000,2,7000,High
105,Vikram Singh,Chennai,Neurology,7000,1,8000,High
106,Ananya Das,Kolkata,Orthopedics,3000,3,6000,Normal
107,Karan Patel,Ahmedabad,Cardiology,5000,1,6000,Normal
108,Meera Iyer,Bangalore,Dermatology,1500,2,3500,Normal
109,Rohit,Pune,Neurology,6500,2,8500,High


When we use INSERT,UPDATE,DELETE,MERGE does not immediately remove old data files.
Instead, Delta Lake keeps old files temporarily so that features like:
      Time Travel
      Version History
      Rollback
can work properly.
Because of this, many unused old files remain in storage.
Vaccum is used to clean and clear the old and unneccesary files,removes files that are no longer used by the delta tables
 

In [0]:
%sql
VACUUM hospital_delta RETAIN 168 HOURS DRY RUN;

path


In [0]:

spark.sql("SELECT * FROM patient_delta").show()
spark.sql("DESCRIBE DETAIL patient_delta").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+

+------+--------------------+--------------------+-----------+--------+--------------------+-------

In [0]:

%sql
CREATE TABLE patient_target
USING DELTA
AS
SELECT * FROM hospital_visits;
     

num_affected_rows,num_inserted_rows


In [0]:

%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES(101,"Arjun Reddy","Hyderabad","Cardiology",6000,2,7000,"High"), 
(112,"Pooja Shah","Mumbai","Neurology",8000,1,8500,"High") 
AS daily_updates(visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_type);

In [0]:

%sql
MERGE INTO patient_target AS target
USING daily_updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.total_bill = source.total_bill,
target.patient_type = source.patient_type

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1
